# 05 · Drought penalty without trend fitting (Iowa corn and soybeans)

**Direction:** Soil and Crop Resilience Planning. **User:** Soil Conservation Office.

Notebook 04 showed two problems with a trend-based anomaly: a 2000–2015 linear trend extrapolates with errors that correlate with soil water storage, and the priority scores "predicted" losses even in years with no drought (placebo failure), which points to persistent county differences rather than drought vulnerability.

This notebook replaces the trend with a **local drought penalty** that cancels persistent county differences:

1. **r** = log(yield) − statewide median log(yield) that year, same crop. Removes the common weather shock.
2. For a D2+ county-year, its **normal** is the mean r over the same county's non-D2+ years within ± `WINDOW` years, **inside the same period** (TRAIN years for TRAIN, TEST years for TEST, so nothing crosses the split). At least `MIN_NORMAL_YEARS` normal years are required.
3. **Drought penalty** = r in the drought year − normal, converted to approximate percent, 100 × (exp(x) − 1). Negative = the county lost more than usual relative to the state.
4. **Placebo** = the same quantity for non-D2+ county-years (their r minus the mean r of their *other* non-D2+ years in the window). It should average about zero and relate to nothing.

Then: the storage association in drought years vs the placebo (Step 2), and a held-out ranking test with its own placebo (Step 3). `yield_anomaly_pct` is not used anywhere here.

**The method lives in `soil_planner/soil_tool.py`** (`compute_penalties`, `penalty_storage_results`, `penalty_ranking_results`) so the deliverable notebook uses exactly the code tested here. The parameters are printed below from that module.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from soil_planner import soil_tool as st
from src.data_loader import load_master

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

# ---- Parameters and column names, read from soil_tool.py so this notebook and the deliverable cannot drift apart ----
WINDOW, MIN_NORMAL_YEARS, SEV_D2 = st.WINDOW, st.MIN_NORMAL_YEARS, st.SEV_D2
TRAIN_YEARS, TEST_YEARS = st.TRAIN_YEARS, st.TEST_YEARS
MIN_COUNTIES_PER_YEAR, MIN_D2_TRAIN_YEARS_FOR_SCORE, TOP_SHARE = st.MIN_COUNTIES_PER_YEAR, st.MIN_D2_YEARS_FOR_SCORE, st.TOP_SHARE
N_BOOT, BOOT_SEED, CI, AWS_UNIT = st.N_BOOT, st.BOOT_SEED, st.CI, st.AWS_UNIT
COL_FIPS, COL_COUNTY, COL_YEAR, COL_CROP, COL_YIELD = st.COL_FIPS, st.COL_COUNTY, st.COL_YEAR, st.COL_CROP, st.COL_YIELD
COL_SEV, COL_AWS, COL_LON, COL_R, COL_PEN, COL_KIND = st.COL_SEV, st.COL_AWS, st.COL_LON, st.COL_R, st.COL_PEN, st.COL_KIND
FOCUS_CROPS = list(st.CROPS.values())
SCORE_DIR = st.PEN_SCORE_DIR

print(f"WINDOW = {WINDOW}, MIN_NORMAL_YEARS = {MIN_NORMAL_YEARS}, D2 code = {SEV_D2}, "
      f"TRAIN = {TRAIN_YEARS.start}-{TRAIN_YEARS.stop - 1}, TEST = {TEST_YEARS.start}-{TEST_YEARS.stop - 1}, "
      f"min counties per year = {MIN_COUNTIES_PER_YEAR}, min D2+ train years for a score = {MIN_D2_TRAIN_YEARS_FOR_SCORE}, "
      f"bootstrap draws = {N_BOOT}, seed = {BOOT_SEED}")

WINDOW = 3, MIN_NORMAL_YEARS = 2, D2 code = 3, TRAIN = 2000-2015, TEST = 2016-2025, min counties per year = 15, min D2+ train years for a score = 3, bootstrap draws = 5000, seed = 20261017


## Data and Step 1 — drought penalty and placebo (split: normals never cross 2015/2016)

In [2]:
ia = st.compute_penalties(st.load_iowa_with_lon(), split=True)   # Iowa corn + soybeans, reported yields, fips as text
assert ia[COL_FIPS].map(type).eq(str).all() and ia[COL_FIPS].str.len().eq(5).all()
print(ia.groupby([COL_CROP, "period"]).agg(rows=(COL_FIPS, "size"), counties=(COL_FIPS, "nunique"), d2_rows=("is_d2", "sum")))

drop = (ia.groupby([COL_CROP, "period", COL_KIND])
          .agg(county_years=(COL_FIPS, "size"), with_value=(COL_PEN, lambda s: s.notna().sum()))
          .assign(dropped=lambda d: d["county_years"] - d["with_value"],
                  pct_dropped=lambda d: (100 * d["dropped"] / d["county_years"]).round(1)))
print(f"\nCounty-years dropped for fewer than {MIN_NORMAL_YEARS} normal years in a +/- {WINDOW}-year window (same period):")
display(drop)

                 rows  counties  d2_rows
crop     period                         
CORN     test     890        99      300
         train   1580        99      462
SOYBEANS test     855        99      335
         train   1582        99      501

County-years dropped for fewer than 2 normal years in a +/- 3-year window (same period):


county_years  with_value  dropped  pct_dropped
crop     period kind                                                   
CORN     test   drought           300         277       23          7.7
                placebo           590         479      111         18.8
         train  drought           462         449       13          2.8
                placebo          1118        1026       92          8.2
SOYBEANS test   drought           335         263       72         21.5
                placebo           520         402      118         22.7
         train  drought           501         482       19          3.8
                placebo          1081         982       99          9.2

In [3]:
# Sanity: the placebo should average about zero; the drought penalty should be negative on average.
print("Mean and spread of the penalty (approx %), by crop and kind:")
display(ia.groupby([COL_CROP, COL_KIND])[COL_PEN].describe()[["count", "mean", "std", "25%", "50%", "75%"]].round(2))

print("\nMean drought penalty and placebo by year (counties with a value in brackets):")
by_year = (ia.groupby([COL_CROP, COL_YEAR, COL_KIND])[COL_PEN]
             .agg(["mean", "count"]).round(1).unstack(COL_KIND))
by_year.columns = [f"{k}: {m}" for m, k in by_year.columns]
display(by_year.unstack(COL_CROP).swaplevel(axis=1).sort_index(axis=1))

Mean and spread of the penalty (approx %), by crop and kind:


count  mean    std   25%   50%   75%
crop     kind                                          
CORN     drought   726.0 -2.03  11.18 -7.81 -1.28  4.43
         placebo  1505.0  0.36   8.54 -4.23  0.80  5.14
SOYBEANS drought   745.0 -1.73   9.78 -7.24 -1.30  4.40
         placebo  1384.0  0.35   7.48 -3.96  0.55  4.88


Mean drought penalty and placebo by year (counties with a value in brackets):


crop           CORN                                                  SOYBEANS                                           
     drought: count drought: mean placebo: count placebo: mean drought: count drought: mean placebo: count placebo: mean
year                                                                                                                    
2000           70.0           1.4           16.0          -7.0           70.0           0.8           16.0           5.3
2001            NaN           NaN           86.0           1.4            NaN           NaN           86.0          -0.1
2002           14.0         -12.7           85.0          -0.3           14.0          -8.8           85.0          -0.9
2003           92.0          -2.9            7.0           1.8           92.0          -1.9            7.0          11.3
2004            NaN           NaN           99.0          -0.4            NaN           NaN           99.0          -0.7
2005           26.0         -15.4           73.0           2.8           47.0           0.4           52.0           2.3
2006           51.0          -2.4           48.0           3.1           51.0          -0.2           48.0           1.8
2007           10.0          -8.1           89.0           3.7           10.0          -0.0           89.0           1.3
2008            NaN           NaN           99.0           0.0            NaN           NaN           99.0          -2.7
2009            NaN           NaN           99.0           3.2            NaN           NaN           99.0          -0.3
2010            NaN           NaN           99.0          -7.1            NaN           NaN           99.0           1.2
2011           16.0         -10.8           83.0          -0.4           33.0          -3.4           66.0           0.6
2012           99.0           0.9            NaN           NaN           99.0          -1.6            NaN           NaN
2013           64.0           3.5           35.0           4.8           65.0          -0.6           34.0           1.8
2014            7.0           0.6           74.0           1.3            1.0           7.9           70.0           1.1
2015            NaN           NaN           34.0          -0.9            NaN           NaN           33.0           1.1
2016            NaN           NaN           81.0          -0.9            NaN           NaN           79.0           0.4
2017           26.0          -7.9           69.0           2.2           30.0          -6.3           65.0           0.5
2018           14.0           4.0           78.0          -0.3           17.0           3.8           77.0          -0.6
2019            NaN           NaN           88.0          -1.7            NaN           NaN           80.0          -0.2
2020           48.0          -3.1           41.0           3.0           42.0          -1.9           37.0           1.1
2021           57.0          -0.7           25.0          -3.3           50.0           0.4           18.0          -1.1
2022           41.0          -5.1           47.0           3.1           50.0          -6.9           31.0           1.5
2023           73.0          -0.8            1.0           6.4           52.0          -2.8            1.0           0.6
2024           18.0          -3.8           30.0           1.4           22.0          -3.7           10.0           4.1
2025            NaN           NaN           19.0          -3.2            NaN           NaN            4.0          -1.1

In [4]:
fig = px.box(ia[ia[COL_PEN].notna()], x=COL_YEAR, y=COL_PEN, color=COL_KIND, facet_row=COL_CROP,
             color_discrete_map={"drought": "#d7301f", "placebo": "#9ecae1"},
             title="Drought penalty (red) and placebo (blue) by year: county distribution, approx. % relative to the county's own normal",
             labels={COL_PEN: "penalty (%)", COL_YEAR: "year"})
fig.add_hline(y=0, line_dash="dot", line_color="grey")
fig.update_layout(height=600)
fig.show()

### Does the placebo relate to storage or longitude? (it should not)

In [5]:
plc = ia[(ia[COL_KIND] == "placebo") & ia[COL_PEN].notna()]
cty = plc.groupby([COL_CROP, COL_FIPS]).agg(mean_placebo=(COL_PEN, "mean"), storage=(COL_AWS, "first"), lon=(COL_LON, "first")).reset_index()
print("County mean placebo vs storage and longitude (Pearson), all years:")
display(cty.groupby(COL_CROP).apply(lambda g: pd.Series({
    "corr(mean placebo, storage)": g["mean_placebo"].corr(g["storage"]),
    "corr(mean placebo, longitude)": g["mean_placebo"].corr(g["lon"]),
    "mean placebo": g["mean_placebo"].mean(), "sd across counties": g["mean_placebo"].std()}),
    include_groups=False).round(3))

County mean placebo vs storage and longitude (Pearson), all years:


,"corr(mean placebo, storage)","corr(mean placebo, longitude)",mean placebo,sd across counties
crop,,,,
CORN,-0.360,0.004,0.386,0.705
SOYBEANS,0.001,-0.108,0.323,0.553


---
## Step 2 — Storage and the drought penalty, in-sample, all years

Within each year with at least 15 D2+ counties, regress the drought penalty on storage (per +10 mm), with and without longitude. Average the per-year slopes; 95% range from resampling years. Same thing on the placebo in each year with at least 15 non-D2+ counties with a placebo value. A real buffering effect would appear in the drought column and not in the placebo column.

In [6]:
step2, slopes = st.penalty_storage_results(ia)
display(step2.drop(columns=["_mean", "_lo", "_hi"])
             .pivot(index=["crop", "spec"], columns="kind",
                    values=["estimate [95% range]", "n_years", "county_years", "years +/-", "excludes_zero"])
             .swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)[["drought", "placebo"]])
print("Units: percentage points of penalty per +10 mm of aws_100cm_mm. Positive = more storage, smaller penalty.")

kind                                       drought                                                            placebo                                             
                              estimate [95% range] n_years county_years years +/- excludes_zero  estimate [95% range] n_years county_years years +/- excludes_zero
crop     spec                                                                                                                                                     
CORN     storage + longitude  +0.88 [-1.72, +3.75]      13          681     7 / 6         False  -0.24 [-1.09, +0.64]      23         1497   10 / 13         False
         storage only         +0.83 [-1.57, +3.44]      13          681     9 / 4         False  -0.30 [-1.13, +0.51]      23         1497    8 / 15         False
SOYBEANS storage + longitude  +1.38 [-0.92, +3.38]      14          720     9 / 5         False  +0.13 [-0.93, +1.23]      21         1362   11 / 10         False
         storage only         +1.81 [-0.08, +3.73]      14          720     9 / 5         False  -0.30 [-1.26, +0.60]      21         1362   10 / 11         False

Units: percentage points of penalty per +10 mm of aws_100cm_mm. Positive = more storage, smaller penalty.


In [7]:
print("Per-year slopes, storage + longitude:")
for crop in FOCUS_CROPS:
    t = slopes[(crop, "drought", "storage + longitude")].rename(columns=lambda c: f"drought {c}") \
        .join(slopes[(crop, "placebo", "storage + longitude")].rename(columns=lambda c: f"placebo {c}"), how="outer")
    print(crop)
    display(t.round(2).T)

Per-year slopes, storage + longitude:
CORN


year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
drought n_counties,70.00,NaN,NaN,92.00,NaN,26.00,51.00,NaN,NaN,NaN,NaN,16.00,99.00,64.00,NaN,NaN,NaN,26.00,NaN,NaN,48.00,57.00,41.00,73.00,18.00,NaN
drought slope_per_10mm,-5.06,NaN,NaN,2.13,NaN,-3.53,-3.13,NaN,NaN,NaN,NaN,14.26,7.57,0.58,NaN,NaN,NaN,2.77,NaN,NaN,-5.49,-0.49,2.08,-1.09,0.85,NaN
placebo n_counties,16.00,86.00,85.00,NaN,99.00,73.00,48.00,89.00,99.00,99.0,99.00,83.00,NaN,35.00,74.00,34.00,81.00,69.00,78.00,88.00,41.00,25.00,47.00,NaN,30.00,19.0
placebo slope_per_10mm,1.60,1.77,0.03,NaN,-0.69,-2.39,-1.56,-4.45,3.54,-1.8,4.78,-2.42,NaN,0.94,-1.59,-1.86,-0.86,0.14,3.25,-0.41,0.54,-1.00,0.54,NaN,-0.83,-2.8


SOYBEANS


year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
drought n_counties,70.00,NaN,NaN,92.00,NaN,47.00,51.00,NaN,NaN,NaN,NaN,33.00,99.00,65.00,NaN,NaN,NaN,30.0,17.00,NaN,42.00,50.00,50.00,52.0,22.00
drought slope_per_10mm,-7.13,NaN,NaN,3.65,NaN,-5.87,-2.65,NaN,NaN,NaN,NaN,4.66,2.69,5.88,NaN,NaN,NaN,3.5,6.22,NaN,-2.18,-0.03,5.42,2.1,3.11
placebo n_counties,16.00,86.00,85.00,NaN,99.0,52.00,48.00,89.00,99.00,99.00,99.00,66.00,NaN,34.00,70.0,33.00,79.00,65.0,77.00,80.00,37.00,18.00,31.00,NaN,NaN
placebo slope_per_10mm,2.86,2.48,0.75,NaN,-2.5,0.38,-0.66,-3.78,4.48,1.47,-1.65,-1.53,NaN,5.58,0.4,-2.91,-2.22,-0.0,2.13,0.08,2.64,-3.10,-2.15,NaN,NaN


---
## Step 3 — Held-out ranking test (scores from TRAIN, tested on TEST drought years)

Training scores per county: `penalty_score` = mean drought penalty over the county's D2+ TRAIN years (needs ≥ 3), `drought_freq` = share of TRAIN years at D2+, `storage` = `aws_100cm_mm`. Priority is oriented so higher = more deserving: more negative penalty, higher frequency, lower storage.

For each TEST year with ≥ 15 D2+ counties: Spearman between priority and that year's drought penalty, signed so positive = the score correctly picks the counties that lost more; and the share of top-third-priority counties that landed in the worst third (chance ≈ 0.33).

**Placebo test:** the identical test on the placebo values, in the TEST years with no D2+ county at all (2016, 2019, 2025), over all reporting counties. A second placebo table uses the non-D2+ counties of every TEST year with ≥ 15 of them. The scores should show nothing in either.

In [8]:
rk = st.penalty_ranking_results(ia)
scores, results, step3 = rk["scores"], rk["results"], rk["summary"]

for crop in FOCUS_CROPS:
    sc = scores[scores[COL_CROP] == crop]
    miss = sc[sc["penalty_score"].isna()]
    print(f"{crop}: {len(sc)} counties; penalty_score missing for {len(miss)} "
          f"(< {MIN_D2_TRAIN_YEARS_FOR_SCORE} D2+ TRAIN years with a penalty value): "
          + ", ".join(f"{n.replace(' County', '')} ({int(k)})" for n, k in zip(miss[COL_COUNTY], miss["n_d2_train"])))
print()
display(scores.groupby(COL_CROP)[["penalty_score", "drought_freq", "storage"]].describe().round(2).T)

print()
for crop, info in rk["test_years"].items():
    print(f"{crop}: TEST years with >= {MIN_COUNTIES_PER_YEAR} D2+ counties with a penalty value: {len(info['usable'])} -> "
          + ", ".join(f"{y} ({n})" for y, n in info["usable"].items())
          + f" | zero-drought TEST years: {info['zero_drought']} | other: "
          + ", ".join(f"{y} ({n})" for y, n in info["other"].items()))
    if len(info["usable"]) < 4:
        print("   *** WARNING: fewer than 4 usable test years. ***")

CORN: 99 counties; penalty_score missing for 5 (< 3 D2+ TRAIN years with a penalty value): Buchanan (2), Clayton (2), Fayette (2), Howard (2), Winneshiek (2)
SOYBEANS: 99 counties; penalty_score missing for 5 (< 3 D2+ TRAIN years with a penalty value): Buchanan (2), Clayton (2), Fayette (2), Howard (2), Winneshiek (2)



crop                   CORN  SOYBEANS
penalty_score count   94.00     94.00
              mean    -2.06     -1.10
              std      5.78      5.57
              min    -18.31    -11.92
              25%     -6.02     -5.33
              50%     -2.01     -1.28
              75%      2.36      3.24
              max     10.42      9.80
drought_freq  count   99.00     99.00
              mean     0.29      0.32
              std      0.08      0.09
              min      0.12      0.12
              25%      0.25      0.25
              50%      0.31      0.31
              75%      0.38      0.38
              max      0.47      0.47
storage       count   99.00     99.00
              mean   179.24    179.24
              std     11.42     11.42
              min    154.25    154.25
              25%    172.00    172.00
              50%    179.00    179.00
              75%    184.78    184.78
              max    205.76    205.76


CORN: TEST years with >= 15 D2+ counties with a penalty value: 6 -> 2017 (26), 2020 (48), 2021 (57), 2022 (41), 2023 (73), 2024 (18) | zero-drought TEST years: [2016, 2019, 2025] | other: 2018 (14)
SOYBEANS: TEST years with >= 15 D2+ counties with a penalty value: 7 -> 2017 (30), 2018 (17), 2020 (42), 2021 (50), 2022 (50), 2023 (52), 2024 (22) | zero-drought TEST years: [2016, 2019, 2025] | other: 


In [9]:
for crop in FOCUS_CROPS:
    print(f"===== {crop} =====")
    display(st.ranking_summary_table(step3, crop))
    v = step3[step3["crop"] == crop]
    print("Real test beats chance (whole range above chance):",
          v[(v["test"] == st.REAL_TEST) & v["range_above_chance"]][["score", "metric"]].values.tolist() or "none")
    print()

===== CORN =====


test                         chance   real: drought years placebo: zero-drought years placebo: non-D2+ counties, all test years n_years (real / placebo-0 / placebo-all)
score         metric                                                                                                                                                    
penalty_score spearman         0.00  +0.22 [+0.13, +0.31]        +0.17 [-0.25, +0.42]                      +0.04 [-0.17, +0.27]                                5 / 3 / 9
              top_third_hit    0.33  +0.45 [+0.36, +0.54]        +0.40 [+0.18, +0.52]                      +0.35 [+0.25, +0.46]                                5 / 3 / 9
drought_freq  spearman         0.00  -0.14 [-0.40, +0.15]        -0.05 [-0.19, +0.07]                      -0.03 [-0.17, +0.12]                                6 / 3 / 9
              top_third_hit    0.33  +0.31 [+0.14, +0.49]        +0.34 [+0.26, +0.41]                      +0.35 [+0.26, +0.43]                                6 / 3 / 9
storage       spearman         0.00  +0.17 [+0.05, +0.29]        +0.11 [+0.01, +0.30]                      -0.01 [-0.14, +0.14]                                6 / 3 / 9
              top_third_hit    0.33  +0.36 [+0.21, +0.46]        +0.29 [+0.17, +0.37]                      +0.30 [+0.25, +0.34]                                6 / 3 / 9

Real test beats chance (whole range above chance): [['penalty_score', 'spearman'], ['penalty_score', 'top_third_hit'], ['storage', 'spearman']]

===== SOYBEANS =====


test                         chance   real: drought years placebo: zero-drought years placebo: non-D2+ counties, all test years n_years (real / placebo-0 / placebo-all)
score         metric                                                                                                                                                    
penalty_score spearman         0.00  +0.16 [-0.01, +0.35]        -0.13 [-0.35, +0.09]                      +0.05 [-0.11, +0.22]                                7 / 2 / 7
              top_third_hit    0.33  +0.49 [+0.42, +0.54]        +0.20 [+0.20, +0.20]                      +0.32 [+0.24, +0.40]                                7 / 2 / 7
drought_freq  spearman         0.00  -0.19 [-0.53, +0.15]        -0.13 [-0.36, +0.10]                      +0.03 [-0.21, +0.27]                                7 / 2 / 7
              top_third_hit    0.33  +0.36 [+0.19, +0.53]        +0.25 [+0.19, +0.31]                      +0.36 [+0.24, +0.49]                                7 / 2 / 7
storage       spearman         0.00  +0.36 [+0.16, +0.56]        -0.01 [-0.04, +0.01]                      -0.02 [-0.16, +0.10]                                7 / 2 / 7
              top_third_hit    0.33  +0.53 [+0.44, +0.61]        +0.30 [+0.27, +0.33]                      +0.28 [+0.19, +0.35]                                7 / 2 / 7

Real test beats chance (whole range above chance): [['penalty_score', 'top_third_hit'], ['storage', 'spearman'], ['storage', 'top_third_hit']]



In [10]:
print("Per-year values, real test on drought years (Spearman: positive = correct ordering):")
for crop in FOCUS_CROPS:
    py, _ = results[(crop, st.REAL_TEST)]
    sp = pd.concat({n: p["spearman"] for n, p in py.items()}, axis=1)
    sp.insert(0, "n_counties", py["drought_freq"]["n"].astype(int))
    hit = pd.concat({n: p["top_third_hit"] for n, p in py.items()}, axis=1).add_prefix("hit: ")
    print(crop)
    display(sp.join(hit).round(2))

print("Per-year values, placebo on zero-drought years:")
for crop in FOCUS_CROPS:
    py, _ = results[(crop, st.PLACEBO_ZERO)]
    sp = pd.concat({n: p["spearman"] for n, p in py.items()}, axis=1)
    sp.insert(0, "n_counties", py["drought_freq"]["n"].astype(int))
    print(crop)
    display(sp.round(2))

Per-year values, real test on drought years (Spearman: positive = correct ordering):
CORN


,n_counties,penalty_score,drought_freq,storage,hit: penalty_score,hit: drought_freq,hit: storage
year,,,,,,,
2017,26,0.34,0.39,0.02,0.38,0.67,0.33
2020,48,0.12,-0.01,0.16,0.31,0.44,0.50
2021,57,0.09,-0.50,0.35,0.47,0.11,0.47
2022,41,0.24,0.20,0.37,0.50,0.43,0.43
2023,73,0.31,-0.43,0.09,0.61,0.25,0.42
2024,18,NaN,-0.47,0.02,NaN,0.00,0.00


SOYBEANS


,n_counties,penalty_score,drought_freq,storage,hit: penalty_score,hit: drought_freq,hit: storage
year,,,,,,,
2017,30,-0.07,0.38,0.36,0.40,0.70,0.50
2018,17,-0.10,-0.17,0.76,0.50,0.33,0.67
2020,42,0.65,0.17,-0.14,0.50,0.57,0.29
2021,50,0.02,-0.61,0.34,0.33,0.12,0.47
2022,50,0.38,0.31,0.26,0.53,0.53,0.59
2023,52,0.11,-0.68,0.25,0.56,0.12,0.65
2024,22,0.12,-0.73,0.72,0.57,0.14,0.57


Per-year values, placebo on zero-drought years:
CORN


,n_counties,penalty_score,drought_freq,storage
year,,,,
2016,81,0.35,-0.19,0.01
2019,88,-0.25,0.07,0.30
2025,19,0.42,-0.03,0.02


SOYBEANS


,n_counties,penalty_score,drought_freq,storage
year,,,,
2016,79,0.09,0.10,-0.04
2019,80,-0.35,-0.36,0.01


In [11]:
fig = go.Figure()
short = {st.PLACEBO_ALL: "placebo-all", st.PLACEBO_ZERO: "placebo-0", st.REAL_TEST: "real"}
for crop in FOCUS_CROPS:
    v = step3[(step3["crop"] == crop) & (step3["metric"] == "spearman")].copy()
    v["x"] = v["score"] + " · " + v["test"].map(short)
    fig.add_trace(go.Scatter(x=v["x"], y=v["_mean"], mode="markers", name=crop, marker=dict(size=9),
                             error_y=dict(type="data", symmetric=False, array=v["_hi"] - v["_mean"], arrayminus=v["_mean"] - v["_lo"])))
fig.add_hline(y=0, line_dash="dot", line_color="grey", annotation_text="chance")
fig.update_layout(title="Held-out Spearman (positive = correct) with 95% year-bootstrap range: real test vs placebos",
                  yaxis_title="Spearman", height=450, xaxis_tickangle=-30)
fig.show()

---
## Reading the results (plain language)

**The penalty behaves as intended.** The placebo averages about +0.4 pp for both crops with a within-county spread of 8 pp, and the drought penalty averages about −2 pp. County-mean placebos vary by less than 1 pp across counties. For corn that small county-mean placebo still correlates −0.36 with storage, which is worth remembering, but the year-level placebo slope on storage is indistinguishable from zero (−0.24 [−1.09, +0.64] per 10 mm with longitude). Drop rates are low in TRAIN (3–4% of drought county-years) and higher in TEST (8% corn, 22% soybeans), because fewer counties report yields after 2022, so there are fewer normal years to compare against. 2024 corn keeps only 18 drought counties.

**Step 2 — storage and the drought penalty, all years.** With longitude as a control: corn +0.88 [−1.72, +3.75] pp per +10 mm in drought years vs −0.24 [−1.09, +0.64] in the placebo; soybeans +1.38 [−0.92, +3.38] vs +0.13 [−0.93, +1.23]. Without longitude, soybeans reach +1.81 [−0.08, +3.73]. So the association leans positive in drought years and sits at zero in the placebo, which is the pattern a real buffering effect would produce, but every drought-year range still includes zero. The per-year slopes are mixed in the 2000–2015 drought years (2000, 2005 and 2006 negative; 2011, 2012 and 2013 positive) and mostly positive in 2016–2025. This is suggestive of a small storage buffer, more so for soybeans, and it is not demonstrated.

**Step 3 — held-out ranking.** Scores built on 2000–2015, tested on 2016–2025 drought years (corn 6 years, 5 for the penalty score; soybeans 7).
- `penalty_score` (mean drought penalty in TRAIN drought years) is the first score in this project to beat chance. Corn: Spearman +0.22 [+0.13, +0.31], top-third hit rate 0.45 [0.36, 0.54], positive in all 5 usable years. Soybeans: Spearman +0.16 [−0.01, +0.35] (range just touches zero), hit rate 0.49 [0.42, 0.54], positive in 5 of 7 years.
- `storage` alone also ranks test drought years correctly: corn +0.17 [+0.05, +0.29], soybeans +0.36 [+0.16, +0.56], hit rates 0.36 and 0.53. Storage is a fixed attribute, so nothing was trained; this says only that in 2016–2025 drought years higher-storage counties had smaller penalties. Step 2 shows that was not consistently true in 2000–2015, so treat it as period-specific until a longer record exists.
- `drought_freq` does not work: −0.14 [−0.40, +0.15] corn, −0.19 [−0.53, +0.15] soybeans. How often a county was in drought says nothing about how much it loses when it is.

**Does the placebo pass now?** Mostly, with a caveat.
- The broad placebo (non-D2+ counties in every test year, 9 corn years and 7 soybean years) is clean: every score sits at chance (Spearman between −0.03 and +0.05, hit rates 0.28–0.36). The persistent-county effect that broke notebook 04 is gone.
- The strict placebo (years with no drought anywhere: 2016, 2019, 2025) is too thin to judge: 3 years for corn, 2 for soybeans, and 2025 has only 19 corn counties with a value. Its ranges are unreliable (a two-year hit-rate range of [0.20, 0.20] is not a range). Within that thin set the corn penalty score has a positive mean (+0.17 [−0.25, +0.42]) and the corn storage score a range that just clears zero (+0.11 [+0.01, +0.30]); both rest on three numbers and should temper confidence rather than be reported.

**Is the test too thin?** Five to seven test years is enough to say the penalty score did not fail, and that it beat chance on these years. It is not enough to call the ranking reliable. The 2016–2025 droughts were all partial-state events; no statewide drought like 2012 is in the test set, and that is the case the office cares most about.

**What this supports.** A county's past drought penalty, measured against its own nearby normal years, carries some information about which counties lose more in later droughts, for corn and probably for soybeans. Soil water storage leans in the expected direction in drought years and not in the placebo, which is consistent with a small buffering effect, but the ranges include zero and the pattern is weaker before 2016. This remains an association across counties. It is not a measured effect of any soil program, and nothing here estimates what a program would change.
